# SPINE-GPE v7 — PNADc Certifier v1.2.0

Este notebook executa apenas a camada de golden tests SIDRA sobre os Parquets já produzidos. Ele não relê os TXT fixed-width.


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
SCRIPT_NAME = "SPINE_GPEv7_PNADC_CERTIFIER_v1.2.0.py"

candidates = [
    Path("/content") / SCRIPT_NAME,
    ROOT / "scripts" / SCRIPT_NAME,
]

SCRIPT = next((path for path in candidates if path.exists()), None)

print("ROOT:", ROOT)
print("SCRIPT:", SCRIPT)

if not ROOT.exists():
    raise FileNotFoundError(f"Raiz não encontrada: {ROOT}")
if SCRIPT is None:
    raise FileNotFoundError(
        "Envie o script para /content ou coloque-o em SPINE-GPEv7/scripts/."
    )


ROOT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
SCRIPT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.2.0.py


In [3]:
import importlib
import subprocess
import sys

packages = {
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.2",
    "pyarrow": "pyarrow>=17",
    "scipy": "scipy>=1.12",
    "requests": "requests>=2.31",
    "urllib3": "urllib3>=2.2",
    "bs4": "beautifulsoup4>=4.12",
    "openpyxl": "openpyxl>=3.1",
    "xlrd": "xlrd>=2.0",
    "lxml": "lxml>=5",
}

missing = []
for module_name, package_name in packages.items():
    try:
        importlib.import_module(module_name)
        print("OK:", module_name)
    except ImportError:
        missing.append(package_name)

if missing:
    print("Instalando:", missing)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--prefer-binary",
            *missing,
        ],
        check=True,
    )

print("Ambiente pronto.")


OK: numpy
OK: pandas
OK: pyarrow
OK: scipy
OK: requests
OK: urllib3
OK: bs4
OK: openpyxl
OK: xlrd
OK: lxml
Ambiente pronto.


In [4]:
import json
import py_compile

py_compile.compile(str(SCRIPT), doraise=True)
print("py_compile: OK")

phase0_lock = ROOT / "00_admin" / "PHASE0_LOCK.json"
phase0 = json.loads(phase0_lock.read_text(encoding="utf-8"))
print("PHASE0:", phase0.get("status"))

if phase0.get("status") != "RELEASED":
    raise RuntimeError("A Fase 0 estrutural não está RELEASED.")


py_compile: OK
PHASE0: RELEASED


In [5]:
required = [
    ROOT / "03_processed/10_pnadc_certified/certified_pnadc_platform_2022.parquet",
    ROOT / "03_processed/10_pnadc_certified/certified_pnadc_platform_2024.parquet",
    ROOT / "00_admin/registry/certified_pnadc_platform_2022_manifest.json",
    ROOT / "00_admin/registry/certified_pnadc_platform_2024_manifest.json",
]

snapshots = ROOT / "02_interim/10_pnadc_certification/sidra_snapshots"
required_tables = [9432, 9441, 9442, 9443, 9642]

for path in required:
    print(path.exists(), path)
    if not path.exists():
        raise FileNotFoundError(path)

for table in required_tables:
    matches = list(snapshots.glob(f"sidra_{table}_*.csv"))
    print(f"SIDRA {table}:", matches)
    if not matches:
        raise FileNotFoundError(f"Snapshot SIDRA {table} ausente")

print("SIDRA 9518 opcional:", list(snapshots.glob("sidra_9518_*.csv")))


True /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_processed/10_pnadc_certified/certified_pnadc_platform_2022.parquet
True /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_processed/10_pnadc_certified/certified_pnadc_platform_2024.parquet
True /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/registry/certified_pnadc_platform_2022_manifest.json
True /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/registry/certified_pnadc_platform_2024_manifest.json
SIDRA 9432: [PosixPath('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/sidra_snapshots/sidra_9432_platform_any.csv')]
SIDRA 9441: [PosixPath('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/sidra_snapshots/sidra_9441_platform_type.csv')]
SIDRA 9442: [PosixPath('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/sidra_snapshots/sidra_9442_income.csv')]
SIDRA 9443: [PosixPath('/con

In [6]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(SCRIPT),
    "--root",
    str(ROOT),
    "--mode",
    "sidra-only",
    "--sidra-source",
    "cache",
    "--strict",
]

print(" ".join(cmd))

result = subprocess.run(
    cmd,
    text=True,
    capture_output=True,
    check=False,
)

print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)
print("Exit code:", result.returncode)

if result.returncode not in {0}:
    raise RuntimeError(
        "Execução bloqueada. Consulte critical_failures no lock."
    )


/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.2.0.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --mode sidra-only --sidra-source cache --strict
STDOUT:
2026-07-20 18:11:25,184 | INFO | SPINE-GPE PNADc Certifier v1.2.0 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=sidra-only
2026-07-20 18:11:27,405 | INFO | Testes estruturais reutilizados de /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/registry/pnadc_certification_tests_20260720T153400Z.json
2026-07-20 18:11:28,235 | INFO | Recalculando estimativas nacionais 2022 a partir de 13 colunas
2026-07-20 18:11:38,721 | INFO | Recalculando estimativas nacionais 2024 a partir de 13 colunas
2026-07-20 18:11:49,584 | INFO | Usando snapshot SIDRA determinístico: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/sidra_snapshots/sidra_9432_platform_any.csv
2026-07-20 18:11:49,918 | INFO | Usand

In [7]:
import json

lock_path = ROOT / "00_admin" / "PNADC_CERTIFICATION_LOCK.json"
lock = json.loads(lock_path.read_text(encoding="utf-8"))

print(json.dumps(lock, ensure_ascii=False, indent=2))

status = lock.get("status")
print("STATUS:", status)

if status == "BLOCKED":
    raise RuntimeError("Há gate crítico bloqueado.")

if status == "CORE_CERTIFIED":
    print(
        "Núcleo PNADc direto certificado. "
        "Rendimento real e/ou informalidade permanecem pendências secundárias."
    )
elif status == "CERTIFIED":
    print("PNADc direta integralmente certificada.")


{
  "run_id": "20260720T181124Z",
  "script_version": "1.2.0",
  "data_schema_version": "spine-gpe-v7-pnadc-certified-1.1.0",
  "validation_schema_version": "spine-gpe-v7-pnadc-validation-1.2.0",
  "mode": "sidra-only",
  "status": "CORE_CERTIFIED",
  "critical_failures": [],
  "secondary_failures": [
    {
      "test_id": "golden.2022.platform_any_income",
      "status": "FAIL",
      "severity": "high",
      "year": 2022,
      "message": "Diferença relativa microdado × SIDRA = 2.5863%.",
      "observed": 2884.4188157417207,
      "expected": 2961.0,
      "tolerance": 0.02,
      "evidence": {
        "table": 9442,
        "table_key": "income",
        "source": "cache",
        "contract": "platform_any_income",
        "selectors": {
          "Nível Territorial (Código)": "1",
          "Brasil (Código)": "1",
          "Variável (Código)": "12910",
          "Unidade de Medida (Código)": "38",
          "Nível de instrução (Código)": "120704",
          "Trabalho por meio 

In [8]:
from pathlib import Path

report_path = Path(lock["report"])
print("Relatório:", report_path)

report = report_path.read_text(encoding="utf-8")
print(report[:12000])


Relatório: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/06_reports/pnadc_certification/pnadc_certification_report_20260720T181124Z.md
# SPINE-GPE v7 — Relatório de Certificação PNADc Plataformas

- Run ID: `20260720T181124Z`
- Versão: `1.2.0`
- Data schema: `spine-gpe-v7-pnadc-certified-1.1.0`
- Validation schema: `spine-gpe-v7-pnadc-validation-1.2.0`
- Modo: `sidra-only`
- Status: **CORE_CERTIFIED**
- Raiz: `/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7`
- Gerado em UTC: `2026-07-20T18:11:51.068179+00:00`

## Princípio de identificação

`S140093` mede o uso observado de aplicativo de entrega. A classificação oficial de entrega plataformizada exige `SD14001=1` e `S140093=1`. Nenhuma proxy preenche essas variáveis.

## Fontes e layouts selecionados

| Ano | Trimestre | TXT | Largura | Layout | Parser | SHA-256 TXT |
|---:|---:|---|---:|---|---|---|
| 2022 | 4 | `/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/10_ibge/platform_direct_supplements/2022q4/PNADC